In [2]:
import os
import re
import sys
import json
import torch
import spacy
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

from collections.abc import Mapping, Sequence
import statistics

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/cupy/_environment.py:406: UserWarning: 
cudnn library could not be loaded.

Reason: ImportError (libcudnn.so.8: cannot open shared object file: No such file or directory)

You can install the library by:

  $ python -m cupyx.tools.install_library --library cudnn --cuda 12.x

  warnings.warn(msg)
/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/chainer/backends/cuda.py:154: UserWarning: cuDNN is not enabled.
Please reinstall CuPy after you inst

In [ ]:
# https://chatgpt.com/s/t_69b2c7f7e28081919384bb842e7c46b3 - apie sakiniu preprocesinga kuris sutrumpina 40-60 proc sakiniu ilgio

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1, # Galima priskirti 2 qubitus, jei, pvz, treniravimo rezultatai yra prasti
     }, # AtomicType.PREPOSITIONAL_PHRASE: 0,
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    rewriter = Rewriter([
        'determiner',
        'auxiliary',
        'connector',
        'coordination',
        'prepositional_phrase', 
        # 'subject_rel_pronoun', # They don't hurt performance, and they act as a safety
        # 'object_rel_pronoun'   # net in case the spaCy parser misses a relative clause.
    ])
    
    rewriter.add_rules(conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True  ,     # turn on NVIDIA cuStateVec kernels
    # batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    # batched_shots_gpu_max_qubits=29, # 8 ⋅ 2^n == 7 ⋅ 2^30, n ~= 29.8
    # num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

# comp_pass = backend.default_compilation_pass(2)

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [ ]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def get_deep_shape(obj, level=0):
    indent = "  " * level
    
    # 1. Atomic types (Strings/Bytes) - Check these first 
    # because they are technically Sequences too!
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Dictionaries (Mappings)
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            # Get child shape and strip only the first line's indentation 
            # so we can prefix it with our key label
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 3. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        name = type(obj).__name__
        if not obj:
            return f"{indent}{name}(len=0)"

        header = f"{indent}{name}(len={len(obj)})"
        
        # Calculate shapes of all children
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 4. Base objects (int, float, None, etc.)
    else:
        return f"{indent}{type(obj).__name__}"

def find_mismatches(data_a: List, data_b: List):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

def diagnose_variable(variable):
    # print("TOP LEVEL LENGTH:", len(variable) if hasattr(variable, '__len__') else "N/A")
    print("DEEP TYPE:")
    print(get_deep_type(variable))
    print("\nDEEP SHAPE:")
    print(get_deep_shape(variable))

def print_n_qubits(n_qubits_list: List[int]):
    print(f"qubit_list: {n_qubits_list}")
    print(f"qubit_list_length: {len(n_qubits_list)}")
    print(f"qubit_list_mean: {statistics.mean(n_qubits_list)}\n")
    


In [ ]:

def load_PreSumm_pts(file_number: int=0, ds_purpose: str="train", calculate_articles: bool=False) -> List[Dict] | Tuple[List[Dict], List[int], List[int], List[int]]:
    valid_ds, elements_valid_labels = [], []
    n_sentences, n_labels, n_labels_true = [], [], []
    
    print(f"load_PreSumm_pts_{file_number}")
    file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{file_number}.bert.pt"
    loaded_data = torch.load(file_path)
    
    elements_valid_labels = remove_negative_articles(loaded_data, "src_txt", "src_sent_labels", 1)

    for i, file_article in enumerate(elements_valid_labels):
        quantum_state_distribution_labels = []
        for label in file_article["src_sent_labels"]:
            if label:
                quantum_state_distribution_labels.append([0,1])
            else:
                quantum_state_distribution_labels.append([1,0])

        valid_ds.append(
            {
                "text_sentences": file_article["src_txt"],
                # "org_labels": line["src_sent_labels"],
                "labels": quantum_state_distribution_labels
            }
        )
        n_sentences.append(len(file_article["src_txt"]))
        n_labels.append(len(file_article["src_sent_labels"]))
        n_labels_true.append(file_article["src_sent_labels"].count(1))

    if calculate_articles:
        n_sentences_sum = sum(n_sentences)
        labes_sum       = sum(n_labels)
        labels_true_sum = sum(n_labels_true)
        print(f"\n n_sentences: {n_sentences_sum} \n n_labels: {labes_sum} (average {labes_sum/len(n_labels):.4f}) \n n_labels_true: {labels_true_sum} (average {labels_true_sum/len(n_labels_true):.4f})")
    
    return valid_ds


In [ ]:
CLEAN_REGEX = re.compile(r"[^\w\s'-]")

# https://chatgpt.com/s/t_69c6b7d3545c8191a4935319498eee59  -> shows and explains how to use spacy to remove `appos`, `acl`, `advcl`, etc. relations
REMOVE_DEPS = {
    "appos",  # appositive
    "acl",    # clausal modifier
    "advcl",  # adverbial clause
    "relcl",  # relative clause
    # "amod",   # Adjectival Modifier (optional if the circuits are still to big)
    # "xcomp",   # open clausal complement | Probably do not use
    # "ccomp"    # clausal complement | Probably do not use

}

# https://chatgpt.com/s/t_69cd50d9e7848191aca0c6ccd42a1ff8 -> expains "ner", "textcat", "lemmatizer", "tagger", "parser"
nlp = spacy.load(
    # "en_core_web_sm",
    "en_core_web_trf",
    disable=["ner", "textcat", "lemmatizer", "tagger"],  # keep only parser
)

def sentence_simplify_spacy(sentences: List[str]) -> Tuple[List[str], List[int]]:
    sentences_simplified = []
    removed_idx = []

    for idx, doc in enumerate(nlp.pipe(sentences, batch_size=128)):
        to_remove = set()

        # Step 1: Identify subordinate clauses to remove
        for token in doc:
            if token.dep_ in REMOVE_DEPS:
                to_remove.update(t.i for t in token.subtree)

        # Step 2: Rebuild sentence
        pruned = "".join(
            token.text_with_ws
            for token in doc
            if token.i not in to_remove
        )

        # Step 3: Clear symbols
        cleaned = clean_sentence(pruned)
        # Step 4: Remove multi-spaces
        cleaned = " ".join(cleaned.split())

        if cleaned:
            sentences_simplified.append(cleaned)
        else:
            removed_idx.append(idx)

    print(f"In pruning lost {len(removed_idx)} out of {len(sentences)} sentences.")
    return sentences_simplified, removed_idx

def clean_sentence(sent: str) -> str:
    return CLEAN_REGEX.sub("", sent)

def remove_negative_articles(dataset: List[Dict], sentences_key: str, labels_key: str,
    positive_label, min_positive_ratio: float = 0.10):
    invalid_ratio, valid_articles = [], []
    one_positive_low_ratio, two_positive_low_ratio = [], []
    zero_positive = []
    label_distribution = {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        ">4": 0
    }

    for i, article in enumerate(dataset):
        sentences = article.get(sentences_key)
        labels = article.get(labels_key)

        if sentences is None or labels is None:
            invalid_ratio.append(i)
            continue

        n_labels = len(labels)

        if len(sentences) != len(labels):
            invalid_ratio.append(i)
            continue

        if n_labels == 0:
            invalid_ratio.append(i)
            continue

        positive_count = labels.count(positive_label)
        positive_ratio = positive_count / n_labels

        if positive_count == 0:
            zero_positive.append(i)
            continue

        elif positive_count in label_distribution:
            label_distribution[positive_count] += 1
        else:
            label_distribution[">4"] += 1
        

        if n_labels > 12 and positive_count == 1:
            one_positive_low_ratio.append(i)
        elif n_labels > 20 and positive_count == 1:
            two_positive_low_ratio.append(i)

        valid_articles.append(article)

    print(f"No+: {len(zero_positive)} | 1+ (>12): {len(one_positive_low_ratio)} | 2+ (>30): {len(two_positive_low_ratio)}")
    print(f"Label distribution: 1: {label_distribution[1]} | 2: {label_distribution[2]} | 3: {label_distribution[3]} | 4: {label_distribution[4]} | >4: {label_distribution['>4']}")
    print(f"Positive labels ratio: {positive_ratio:.4f}")
    print(f"Invalid ratio: {len(invalid_ratio)} out of {len(dataset)} articles.")

    return valid_articles

def load_PreSumm_pts(file_number: int=0, ds_purpose: str="train", calculate_articles: bool=False) -> List[Dict] | Tuple[List[Dict], List[int], List[int], List[int]]:
    valid_ds, elements_valid_labels = [], []
    n_sentences, n_labels, n_labels_true = [], [], []
    
    print(f"load_PreSumm_pts_{file_number}")
    file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{file_number}.bert.pt"
    loaded_data = torch.load(file_path)
    
    elements_valid_labels = remove_negative_articles(loaded_data, "src_txt", "src_sent_labels", 1)

    for i, file_article in enumerate(elements_valid_labels):
        quantum_state_distribution_labels = []
        for label in file_article["src_sent_labels"]:
            if label:
                quantum_state_distribution_labels.append([0,1])
            else:
                quantum_state_distribution_labels.append([1,0])

        valid_ds.append(
            {
                "text_sentences": file_article["src_txt"],
                # "org_labels": line["src_sent_labels"],
                "labels": quantum_state_distribution_labels
            }
        )
        n_sentences.append(len(file_article["src_txt"]))
        n_labels.append(len(file_article["src_sent_labels"]))
        n_labels_true.append(file_article["src_sent_labels"].count(1))

    if calculate_articles:
        n_sentences_sum = sum(n_sentences)
        labes_sum       = sum(n_labels)
        labels_true_sum = sum(n_labels_true)
        print(f"\n n_sentences: {n_sentences_sum} \n n_labels: {labes_sum} (average {labes_sum/len(n_labels):.4f}) \n n_labels_true: {labels_true_sum} (average {labels_true_sum/len(n_labels_true):.4f})")
    
    return valid_ds




In [ ]:
def remove_by_idx(ls: List, remove: List[int]):
    remove_set = set(remove)
    return [x for i, x in enumerate(ls) if i not in remove_set]

def sent2diagrams(sentences: List[str]):
    none_idx = []
    valid_diagrams = []

    diagrams = parser.sentences2diagrams(
        sentences, tokenised=False, suppress_exceptions=True
    )

    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)
        else:
            valid_diagrams.append(diag)

    print(f"In sent2diagrams() lost {len(none_idx)} out of {len(sentences)}.")

    return valid_diagrams, none_idx

def normalize(sentence_diagrams: List[Diagram]):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_exception  = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if d is None:
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"normalize() | {e}")
            drop_exception  += 1
            continue

        diagrams_normalized.append(d)

    print(f"In normalize() lost {drop_rewrite} (rewrite) and {drop_exception } (cup removal) out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: List[Diagram]):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ = ansatz(diagram)
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"quantum_encode() | {e}")
            remove.append(i)

    print(f"In quantum_encode() lost {len(remove)} out of {len(diagrams)}.")
    print(len(errs))

    return encoded_diagrams, remove, errs

def will_train(
    circuits: List,
    qubit_limit: int=28,
    # mem_limit_bytes: int = 7 * 2**30,
    mem_limit_bytes: int=16 * 2**28,
    circuit_depth_limit: int=80,
    gate_limit: int=8000,
):

    valid, invalid_idxs, errs = [], [], []
    n_qubits_list = []

    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            n_qubits = tk_circ.n_qubits
            n_qubits_list.append(n_qubits)

            bytes_per_amplitude = 8  # 64-bit float
            needed_bytes = bytes_per_amplitude * (2 ** n_qubits)
            if n_qubits > qubit_limit:
                raise RuntimeError(f"Too many qubits: {n_qubits} > {qubit_limit}")

            if needed_bytes > mem_limit_bytes:
                raise RuntimeError(
                    f"Needs {needed_bytes / 2**30:.2f} GiB > limit {mem_limit_bytes / 2**30:.2f} GiB"   
                )
            
            # even when samll amount of qubits, there could be a lot of gates and circuit depth
            # which will slow down training process
            if tk_circ.depth() > circuit_depth_limit:
                raise RuntimeError(f"Circuit too deep: {tk_circ.depth()}")
            
            if tk_circ.n_gates > gate_limit:
                raise RuntimeError(f"Circuit contains too many gates: {tk_circ.n_gates}")

            # Even better (adaptive constraint)
            # effective_cost = tk_circ.n_gates * (2 ** n_qubits)
            # if effective_cost > threshold:
            #     reject

            # compiled = backend.get_compiled_circuit(tk_circ)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"will_train() | {e}")
            continue

        valid.append(circ)

    
    print(f"In will_train() lost {len(invalid_idxs)} out of {len(circuits)}.")

    return valid, invalid_idxs, errs, n_qubits_list

def preprocess_and_encode(dataset: List[Dict]) -> Tuple[List[Dict], List[str], List[int]]:
    print(f"Preprocessing and encoding {len(dataset)} articles...")
    encoded_data, errors = [], []

    for i, data_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):

    # for i, data_dict in enumerate(dataset):
            text_sentences = data_dict["text_sentences"].copy()
            labels         = data_dict["labels"].deepcopy()

            n_text_sentences = len(text_sentences)
            n_labels = len(labels)

            sentences_simplified, remove = sentence_simplify_spacy(text_sentences)
            text_sentences               = remove_by_idx(text_sentences, remove)
            labels                       = remove_by_idx(labels, remove)

            diagrams, remove = sent2diagrams(sentences_simplified)
            # diagrams         = remove_by_idx(diagrams, remove)
            text_sentences   = remove_by_idx(text_sentences, remove)
            labels           = remove_by_idx(labels, remove)


            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)


            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            circuits, remove, errs4, n_qubits_list = will_train(circuits)
            text_sentences                         = remove_by_idx(text_sentences, remove)
            labels                                 = remove_by_idx(labels, remove)

            print("init:", n_text_sentences, n_labels,"\nafter:", len(circuits), len(text_sentences), len(labels))
            # n_qubits_list = [0]

            encoded_data.append(
                {
                    "circuits": circuits,
                    "labels": labels,
                    "original_text_sentences": text_sentences,
                }
            )

            errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None

    # errors.append(f"n_sentences: {n_text_sentences} | n_circuits: {len(circuits)} | {n_text_sentences - len(circuits)}")

    return encoded_data, errors, n_qubits_list



In [31]:
ld_ds = load_PreSumm_pts(file_number=0, ds_purpose="train", calculate_articles=True)

load_PreSumm_pts_0
No+: 0 | 1+ (>12): 135 | 2+ (>30): 0
Label distribution: 1: 147 | 2: 536 | 3: 1296 | 4: 0 | >4: 0
Positive labels ratio: 0.1818
Invalid ratio: 22 out of 2001 articles.

 n_sentences: 71022 
 n_labels: 71022 (average 35.8878) 
 n_labels_true: 5107 (average 2.5806)


In [9]:
def encode_save_auto(dataset: List[Dict], n_articles: int, left: int=0):
    right = 0
    while right < n_articles:
        right = min(left + 100, n_articles)
        print(left, "-", right)
        encoded_data, errors, n_qubits_list = preprocess_and_encode(dataset[left:right])

        encoded = {
            "encoded_dataset": encoded_data,
            "errors": errors,
            "n_qubits": n_qubits_list
        }
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{left}_{right}.pkl", 'wb') as file:
            pickle.dump(encoded, file)

        left = right

In [ ]:
encode_save_auto(ld_ds, n_articles, left=100)

In [ ]:
# 14m 27s for the first 101 | without compiled circuit int will_train() | 28 | 200 | 20000

# 101 - 201
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:47<00:00, 20.28s/it]
# 201 - 301
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:48<00:00, 19.09s/it]
# 301 - 401
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:41<00:00, 20.21s/it]
# 401 - 501
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:08<00:00, 18.69s/it]
# 501 - 601
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [35:07<00:00, 21.08s/it]
# 601 - 701
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [32:48<00:00, 19.68s/it]

In [ ]:
def load_encoded_PreSumm_n_m(starting_index = 0, ending_index = 701):
    datasets_list = []
    for i in range(starting_index, ending_index, 100):
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{i}_{i + 100}.pkl", 'rb') as file:
            PreSum_loaded_ds = pickle.load(file)
            datasets_list.append(PreSum_loaded_ds)
    return datasets_list

def load_encoded(filename = "Dataset/Encoded/cnn_dailymail/PreSumm_0_701.pkl"):
    with open(filename, 'rb') as file:
        loaded_encoded_ds = pickle.load(file)

    return loaded_encoded_ds

In [11]:
loaded_encoded_ds = load_encoded()

In [27]:
encoded_dataset = get_combined_encoded_dataset(loaded_encoded_ds)


In [12]:
# print(get_deep_type(encoded_dataset))

encoded_valid_labels = remove_negative_articles(loaded_encoded_ds["encoded_dataset"], "circuits", "labels", [0,1])

No+: 83 | 1+ (>12): 159 | 2+ (>30): 0
Label distribution: 1: 225 | 2: 266 | 3: 126 | 4: 0 | >4: 0
Positive labels ratio: 0.0400
Invalid ratio: 0 out of 700 articles.


In [19]:
# PreSum_0_100.keys()

for i, error_message in enumerate(loaded_encoded_ds["errors"]):
    # if "will_train()" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
    if "bytes" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
        print(f"{i}: {error_message}")

1265: n_sentences: 3762 | n_circuits: 2474 | 1288
2516: n_sentences: 3601 | n_circuits: 2344 | 1257
3721: n_sentences: 3416 | n_circuits: 2198 | 1218
4910: n_sentences: 3473 | n_circuits: 2263 | 1210
6109: n_sentences: 3371 | n_circuits: 2157 | 1214
7370: n_sentences: 3764 | n_circuits: 2485 | 1279
8540: n_sentences: 3503 | n_circuits: 2308 | 1195


In [16]:
import gc

# Delete large temporary variables
# del expensive_tensors

# Force Python to find unreferenced objects
gc.collect()

# Force the GPU to release the cached memory pool
torch.cuda.empty_cache()

In [ ]:
s = ["The president, speaking in Paris, announced sanctions while addressing the media.", 
     "My friend, a well-known scientist, published a paper.", 
     "The book that I read yesterday was fascinating.", 
     "He left the room while talking on the phone.", 
     "The CEO, who was under pressure, resigned after speaking to the board.", 
     "The president of the company announced reforms.", 
     "The cat sat on the mat."] 

expected_result = ["The president announced sanctions", "My friend published a paper", "The book was fascinating", "He left the room", "The CEO resigned", "The president of the company announced reforms", "The cat sat on the mat"]

for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(s[i])
    print(result)
    print(expected_result[i], "\n")

After pruning: lost 0 / 7 sentences.
The president, speaking in Paris, announced sanctions while addressing the media.
The president announced sanctions
The president announced sanctions 

My friend, a well-known scientist, published a paper.
My friend published a paper
My friend published a paper 

The book that I read yesterday was fascinating.
The book was fascinating
The book was fascinating 

He left the room while talking on the phone.
He left the room
He left the room 

The CEO, who was under pressure, resigned after speaking to the board.
The CEO resigned after speaking to the board
The CEO resigned 

The president of the company announced reforms.
The president of the company announced reforms
The president of the company announced reforms 

The cat sat on the mat.
The cat sat on the mat
The cat sat on the mat 



In [ ]:
# s = combine_n_articles(ld_ds[:1], 1)[0]
s = ld_ds[:1][0]['text_sentences']
for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(result)
    print(s[i], "\n")

After pruning: lost 0 / 36 sentences.
he 's on the outside and the view of viktor yanukovych seems to keep getting dimmer
he 's on the outside looking in , and the view of viktor yanukovych seems to keep getting dimmer . 

a news conference friday from his new quarters in southeastern russia underscored just how dim his prospects appear to be
a news conference friday from his new quarters in southeastern russia underscored just how dim his prospects appear to be , as the ousted ukrainian president complained that his host -- and potential benefactor -- was not around and had done little to make his stay more comfortable . 

i consider that russia must and has to act yanukovych told a phalanx of reporters
" i consider that russia must and has to act , " yanukovych told a phalanx of reporters who had assembled in the city of rostov-on-don , near the southwestern border with ukraine and about 700 miles south of moscow . 

he did not specify what actions he was hoping for but made clear wh